# Phase A — thang so sanh nguon prompt

Do xem LLM co thay duoc chuyen gia trong viec viet prompt khong, va **do luon toc
do**: so luot goi Grounding DINO = `1 object + so prompt`, ma DINO chiem 72%
thoi gian (xem `docs/report-phase-b.md` muc 6b).

| Muc | `--prompt-source` | Prompt | Vai tro |
|---|---|---|---|
| P0 | `generic` | `"defect."` | San tuyet doi |
| P1 | `general` | 3 general_prompts | San co san |
| P2b | `llm` | LLM, chi biet ten class | **Dong gop** |
| P2v | `llm` | LLM + anh normal | **Dong gop** |
| P3 | `manual` | 3 general + K manual | Tran (oracle) |

**P3 da chay roi** — chinh la `run_lite2` o Buoc 2. Khong ton them GPU.

**P2b va P2v chua chay duoc**: can `tools/gen_prompts.py` (Task 4 cua plan).
Notebook nay hien chay P0 va P1, von **khong can LLM**.

## Vi sao chay P0 va P1 truoc

Ba cau hoi, tra loi het trong ~8 phut mot class:

1. **Thang co gian khong?** P1 sat P3 thi tien de Phase A yeu — LLM khong con
   may cho de chung minh.
2. **`t_dino` co giam theo so prompt that khong?** P0 dung 1 prompt so voi 6 cua
   P3. Khong giam gan 6 lan thi gia dinh "so luot DINO ty le thuan voi so prompt"
   sai, va toan bo lap luan toc do phai tinh lai.
3. **May moc chay dung chua** — truoc khi dung toi VLM.

Cau hinh khoa o **lite2** (MobileSAM + MobileNetV3 + Grounding DINO), thang Buoc 1.

Notebook khac: `Benchmark_SAA.ipynb` (baseline), `Profiling_SAA_Lite.ipynb`
(Buoc 1), `Benchmark_SAA_Lite2.ipynb` (Buoc 2).

In [ ]:
# Cai dat. An toan khi chay lai nhieu lan.
%cd /content

# Xoa clone cu TRUOC khi clone. Khong co dong nay thi git clone bao
# "destination path already exists", bo qua im lang, va ban chay tiep bang
# code cu ma khong biet.
!rm -rf /content/Segment-Any-Anomaly
!git clone -b dev https://github.com/SyDuc7421/Segment-Any-Anomaly.git
%cd Segment-Any-Anomaly/

# setuptools >= 80 da bo lenh `setup.py develop`, ma pip dung dung lenh do cho
# ban editable khi co --no-build-isolation. Colab nang image len la GroundingDINO
# gay voi "python setup.py develop did not run successfully". Ghim lai truoc.
!pip install -q "setuptools<80" wheel
import setuptools
print('setuptools:', setuptools.__version__)

# Go pin transformers<4.36 cua GroundingDINO. Phai quet CA requirements.txt,
# khong chi *.py: pip doc requirements.txt, va bo sot no thi pip ha transformers
# xuong 4.35 (keo theo huggingface_hub va tokenizers), roi lenh pip cuoi cell
# lai day len 5.x - vong xoay do de lai mot dong conflict gia.
# Code GroundingDINO trong repo nay da duoc va cho transformers 5.x tu truoc.
import re, pathlib

for pattern in ('*.py', '*.txt'):
    for p in pathlib.Path('GroundingDINO').rglob(pattern):
        txt = p.read_text()
        patched = re.sub(r'transformers[^"\'\n]*<4\.\d+(\.\d+)?', 'transformers>=4.41.0', txt)
        if patched != txt:
            p.write_text(patched)
            print('go pin transformers trong', p)

# KHONG dat -q cho hai lenh editable duoi day: day la cho de gay nhat, va
# loi that nam trong phan output ma -q nuot mat.
%cd GroundingDINO/
!pip install -e . --no-build-isolation
%cd ../SAM
!pip install -e .
%cd ..

!pip install -q "transformers>=4.41.0" "supervision>=0.6.0,<0.21.0" \
    opencv-python pycocotools matplotlib onnxruntime onnx ipykernel gradio loguru

# KHONG kiem tra import o day. Ban editable ghi duong dan vao mot file .pth,
# ma .pth chi duoc doc luc interpreter khoi dong - package vua cai xong van
# "khong ton tai" voi kernel dang chay. Kiem tra nam o cell sau lenh restart.
print('\nCai dat xong. Chay cell tiep theo de restart runtime,')
print('roi cell sau do se xac nhan package da cai duoc that.')

In [ ]:
# Restart so updated transformers is loaded from disk
import os
os.kill(os.getpid(), 9)

In [ ]:
# Xac nhan package da cai THAT - chay sau restart, vi ban editable chi hien
# ra voi interpreter khoi dong lai.
%cd /content/Segment-Any-Anomaly
import importlib.util

missing = [m for m in ('groundingdino', 'segment_anything')
           if importlib.util.find_spec(m) is None]

for m in ('groundingdino', 'segment_anything'):
    print(f'{m}: {"THIEU" if m in missing else "OK"}')

if missing:
    raise RuntimeError(
        f'Cai dat that bai: {missing}. DUNG chay tiep - moi class se chet o dong '
        f'import. Doc output pip cua cell 1 de biet la loi setuptools hay loi '
        f'bien dich CUDA extension.'
    )

%cd /content/Segment-Any-Anomaly
%mkdir -p weights
%cd weights
# -nc: co file roi thi bo qua. Khong co no thi chay lai cell se tai lai 2.4 GB
# va de ra sam_vit_h_4b8939.pth.1 - mot ban sao vo dung.
!wget -nc -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
!wget -nc -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
%cd ..

In [ ]:
%cd /content/Segment-Any-Anomaly
%mkdir -p /content/datasets

from google.colab import userdata
import json, pathlib, os

pathlib.Path('/root/.kaggle').mkdir(exist_ok=True)
pathlib.Path('/root/.kaggle/kaggle.json').write_text(json.dumps({
    'username': userdata.get('KAGGLE_USERNAME'),
    'key': userdata.get('KAGGLE_KEY')
}))
!chmod 600 /root/.kaggle/kaggle.json

# Accept dataset terms first at: https://www.kaggle.com/datasets/ipythonx/mvtec-ad
!pip install -q kaggle
!kaggle datasets download -d ipythonx/mvtec-ad -p /content/datasets/ --unzip

os.environ['MVTEC_DIR'] = '/content/datasets'

from datasets import mvtec_classes
present = [c for c in mvtec_classes if os.path.isdir(f'/content/datasets/{c}')]
print(f'MVTec: {len(present)}/15 classes ready:', present)

## Cai MobileSAM

Bat buoc cho cau hinh lite2.

In [ ]:
# MobileSAM cho Lite-1 va Lite-2. No giu nguyen interface SamPredictor nen la
# drop-in that su - SAA/backbones.py khong phai doi gi ngoai ten bien the.
%cd /content/Segment-Any-Anomaly
!pip install -q git+https://github.com/ChaoningZhang/MobileSAM.git
!wget -q -P weights/ https://github.com/ChaoningZhang/MobileSAM/raw/master/weights/mobile_sam.pt

import os, importlib.util

ckpt = 'weights/mobile_sam.pt'
size_mb = os.path.getsize(ckpt) / 1e6 if os.path.exists(ckpt) else 0
print(f'{ckpt}: {"OK, %.0f MB" % size_mb if size_mb > 1 else "THIEU - kiem tra URL"}')
print('mobile_sam:', 'OK' if importlib.util.find_spec('mobile_sam') else 'THIEU')

# Cai them co the keo torch/transformers khac ve. Cho no gay O DAY chu dung de
# gay giua lan profiling.
try:
    from GroundingDINO.groundingdino.models import build_model
    print('GroundingDINO van OK sau khi cai')
except Exception as e:
    print(f'CANH BAO: GroundingDINO gay sau khi cai MobileSAM - {type(e).__name__}: {e}')

## Chon muc va class

In [ ]:
from google.colab import drive
import os

try:
    drive.mount('/content/drive')
except ValueError:
    drive.mount('/content/drive', force_remount=True)

# ---------------------------------------------------------------------------
# Lat cat 4 class cho khoang cach P1 -> P3 chi +0.43 diem p_f1, va prompt thu
# cong THUA tren 2/4 class. Nhung trung binh do bi `carpet` chi phoi (p_f1 55 so
# voi 15-36 cua ba class con lai), nen chua ket luan duoc. Chay ca 15 class.
#
# Doi lai dong duoi de quay ve lat cat nhanh.
# ---------------------------------------------------------------------------
from datasets import mvtec_classes

CLASSES = list(mvtec_classes)
# CLASSES = ['carpet', 'grid', 'metal_nut', 'transistor']   # lat cat ~25 phut

TEXTURE = {'carpet', 'grid', 'leather', 'tile', 'wood'}
LEVELS = ['P0', 'P1']          # them 'P2b', 'P2v' khi Task 4 xong

# (prompt_source, mau duong dan file JSON hoac None)
LEVEL_CONFIG = {
    'P0':  ('generic', None),
    'P1':  ('general', None),
    'P2b': ('llm', 'SAA/prompts/generated/{dataset}-blind.json'),
    'P2v': ('llm', 'SAA/prompts/generated/{dataset}-vision.json'),
    'P3c': ('llm', 'SAA/prompts/generated/{dataset}-manual-as-json.json'),
}

DRIVE_ROOT = '/content/drive/MyDrive/SAA_results'
DATASET = 'mvtec'
os.environ['MVTEC_DIR'] = '/content/datasets'

print('CLASSES =', CLASSES)
print('LEVELS  =', LEVELS)

## Chay thang

In [ ]:
# Goi thang eval_SAA.py chu khong qua run_MVTec.py: dang chay mot tap con class,
# va moi muc can thu muc rieng.
%cd /content/Segment-Any-Anomaly
import os, subprocess, pandas as pd
from collections import deque

def already_done(root, class_name):
    path = f'{root}/csv/{DATASET}-indx-0.csv'
    if not os.path.exists(path):
        return False
    df = pd.read_csv(path, index_col=0)
    return class_name in df.index and df.loc[class_name, 'p_ap'] > 0

for level in LEVELS:
    source, file_template = LEVEL_CONFIG[level]
    root = f'{DRIVE_ROOT}/phase_a_{level}'

    for class_name in CLASSES:
        if already_done(root, class_name):
            print(f'skip {level}/{class_name}: da co ket qua')
            continue

        cmd = [
            'python', 'eval_SAA.py',
            '--dataset', DATASET, '--class-name', class_name,
            '--prompt-source', source,
            '--cal-pro', 'False',
            '--vis', 'False',
            '--sam-variant', 'mobile_sam',
            '--saliency-backbone', 'mobilenetv3',
            '--sam_checkpoint', 'weights/mobile_sam.pt',
            '--detector', 'grounding_dino',
            '--root-dir', root,
        ]
        if file_template:
            path = file_template.format(dataset=DATASET)
            if not os.path.exists(path):
                print(f'skip {level}/{class_name}: chua co {path} (can Task 4)')
                continue
            cmd += ['--llm-prompt-file', path]

        print(f'\n=== {level} / {class_name} ({source}) ===')

        # Doc output trong Python roi print: ipykernel chi bat sys.stdout o muc
        # Python, nen traceback cua tien trinh con co the bien mat hoan toan.
        proc = subprocess.Popen(cmd, cwd='/content/Segment-Any-Anomaly',
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
        tail = deque(maxlen=40)
        for line in proc.stdout:
            print(line, end='')
            tail.append(line)

        if proc.wait() != 0:
            print(f'\n{"=" * 60}\nLOI: {level}/{class_name}. 40 dong cuoi:\n{"=" * 60}')
            print(''.join(tail))
            raise SystemExit(1)

## Chan doan prompt chet (tuy chon, ~4 phut)

Chi can khi ban thay hai muc cho `p_f1` bang nhau.

In [ ]:
# Chan doan: prompt thu cong cua `metal_nut` co dong gop box nao khong?
#
# Lat cat 4 class cho P1 va P3 BANG NHAU den hai chu so (36.13) tren metal_nut.
# P3 = P1 cong dung mot prompt: 'blue defect. black defect. red defect. scratch.'
# Bang nhau nghia la prompt do khong dong gop box nao song sot vao top-k - dung
# thu ma spec muc 5.8 goi la "prompt chet".
#
# Lan chay run_lite2 dien ra TRUOC khi co bo dem (commit 2554aac), nen log cu
# khong co so. Chay lai mot class de lay.
%cd /content/Segment-Any-Anomaly
import subprocess
from collections import deque

CHECK_CLASS = 'metal_nut'
CHECK_ROOT = f'{DRIVE_ROOT}/phase_a_P3check'

proc = subprocess.Popen([
    'python', 'eval_SAA.py',
    '--dataset', DATASET, '--class-name', CHECK_CLASS,
    '--prompt-source', 'manual',
    '--cal-pro', 'False', '--vis', 'False',
    '--sam-variant', 'mobile_sam', '--saliency-backbone', 'mobilenetv3',
    '--sam_checkpoint', 'weights/mobile_sam.pt',
    '--detector', 'grounding_dino',
    '--root-dir', CHECK_ROOT,
], cwd='/content/Segment-Any-Anomaly',
   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

box_lines = []
capturing = False
for line in proc.stdout:
    print(line, end='')
    if 'so box song sot moi prompt' in line:
        capturing = True
    elif capturing and 'box /' in line:
        box_lines.append(line.strip())
    elif capturing and 'box /' not in line and box_lines:
        capturing = False
proc.wait()

print('\n' + '=' * 70)
print(f'So box song sot moi prompt — {CHECK_CLASS}, nguon prompt manual:')
print('=' * 70)
for line in box_lines:
    print(line)
print()
print('Prompt thu cong cua metal_nut la dong chua "blue defect".')
print('0 box tren toan class -> prompt chet, va no giai thich vi sao P1 == P3.')
print('Co box ma p_f1 van bang nhau -> box co sinh ra nhung khong lot top-k')
print(f'(k_mask), mot dang chet khac.')

## Ket qua

In [ ]:
# Doi chieu thang. P3 lay tu lan chay Buoc 2, khong chay lai.
#
# KHONG so cot r_f1 / r_f1_fixed / p_pro: P0-P2 chay --cal-pro False nen chung
# bang 0, con P3 chay voi cal_pro bat. Dat canh nhau la so sanh nham.
import pandas as pd, os, json

COLS = ['p_ap', 'p_f1', 't_dino', 't_total']

from SAA.prompts.mvtec_parameters import manual_prompts as MANUAL


def dino_calls(level, class_name):
    """So luot goi DINO moi anh = 1 object + so prompt defect."""
    if level == 'P0':
        return 1 + 1                       # "defect."
    if level == 'P1':
        return 1 + 3                       # 3 general_prompts
    if level == 'P3':
        return 1 + 3 + len(MANUAL[class_name])   # general + manual
    path = LEVEL_CONFIG[level][1].format(dataset=DATASET)
    if not os.path.exists(path):
        return None
    specs = {s['class']: s for s in json.load(open(path))}
    return 1 + len(specs[class_name]['defect_prompts'])


def read_level(root, level):
    path = f'{root}/csv/{DATASET}-indx-0.csv'
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path, index_col=0)
    df = df[df.index.isin(CLASSES) & (df['p_ap'] > 0)]
    return df if len(df) else None


frames = {lv: read_level(f'{DRIVE_ROOT}/phase_a_{lv}', lv) for lv in LEVELS}
frames['P3'] = read_level(f'{DRIVE_ROOT}/run_lite2', 'P3')
frames = {k: v for k, v in frames.items() if v is not None}

if not frames:
    print('Chua co ket qua nao.')
else:
    common = set.intersection(*(set(df.index) for df in frames.values()))
    common = [c for c in CLASSES if c in common]
    print(f'{len(common)} class chung: {common}\n')

    tab = pd.DataFrame({lv: df.loc[common].mean(numeric_only=True)
                        for lv, df in frames.items()}).T
    print(tab[[c for c in COLS if c in tab]].to_string(float_format='{:.2f}'.format))

    # --- ms moi luot DINO: kiem gia dinh tuyen tinh ---
    print('\n--- t_dino moi luot goi ---')
    print(f'{"muc":5s} {"luot/anh":>9s} {"t_dino":>9s} {"ms/luot":>9s}')
    for lv in tab.index:
        calls = [dino_calls(lv, c) for c in common]
        if any(c is None for c in calls):
            continue
        avg_calls = sum(calls) / len(calls)
        print(f'{lv:5s} {avg_calls:9.2f} {tab.loc[lv, "t_dino"]:9.1f} '
              f'{tab.loc[lv, "t_dino"] / avg_calls:9.1f}')
    print('Cac dong ms/luot xap xi bang nhau = so luot DINO ty le thuan voi so')
    print('prompt. Lech nhieu = gia dinh sai, lap luan toc do phai tinh lai.')

    if 'P3' in tab.index and 'P1' in tab.index:
        print('\n--- muc tieu 1: thang co gian khong ---')
        gap = tab.loc['P3', 'p_f1'] - tab.loc['P1', 'p_f1']
        print(f'khoang cach P1 -> P3: {gap:+.2f} diem p_f1 (trung binh)')
        if abs(gap) < 2:
            print('  CANH BAO: gan bang nhau. Prompt thu cong cua tac gia khong hon')
            print('  general_prompts bao nhieu, nen LLM khong con may cho de chung minh.')

        beaten = [c for c in common if frames['P1'].loc[c, 'p_f1'] > frames['P3'].loc[c, 'p_f1']]
        if beaten:
            print(f'  P1 THANG P3 tren {len(beaten)}/{len(common)} class: {beaten}')
            print('  Tran khong phai P3. LLM co the vuot ca hai, va khung "thu hep')
            print('  khoang cach toi oracle" khong con la cau hoi dung.')
        for lv in ('P2b', 'P2v'):
            if lv in tab.index:
                closed = (tab.loc[lv, 'p_f1'] - tab.loc['P1', 'p_f1']) / gap * 100
                mark = 'DAT' if closed >= 50 else 'chua dat'
                print(f'  {lv}: thu hep {closed:.1f}%  {mark}  (spec muc 7 can >= 50%)')

        # --- texture so voi object: gia thuyet chinh cua lat cat 4 class ---
        print('\n--- khoang cach P1 -> P3 tach theo loai class ---')
        gaps = {}
        for label, members in (('texture', [c for c in common if c in TEXTURE]),
                               ('object ', [c for c in common if c not in TEXTURE])):
            if not members:
                continue
            g = (frames['P3'].loc[members, 'p_f1'].mean()
                 - frames['P1'].loc[members, 'p_f1'].mean())
            gaps[label.strip()] = g
            print(f'{label}: {g:+6.2f} diem  ({len(members)} class: {members})')

        # Chi ket luan khi chenh lech du lon. Duoi 2 diem thi ca hai deu la
        # nhieu, va noi "object gian rong hon" la doc qua vao so lieu.
        if len(gaps) == 2 and max(abs(v) for v in gaps.values()) >= 2:
            wider = max(gaps, key=lambda k: gaps[k])
            print(f'{wider} gian rong hon -> prompt thu cong ma hoa kien thuc that o do.')
        else:
            print('Ca hai deu duoi 2 diem: khong ket luan duoc gi ve texture vs object.')

    print('\n--- tung class, p_f1 ---')
    per_class = pd.DataFrame({lv: df.loc[common, 'p_f1'] for lv, df in frames.items()})
    per_class['loai'] = ['texture' if c in TEXTURE else 'object' for c in per_class.index]
    print(per_class.to_string(float_format='{:.2f}'.format))

    if 'P3' in tab.index:
        print('\n--- muc tieu 2: doi accuracy lay toc do ---')
        for lv in tab.index:
            d_f1 = tab.loc[lv, 'p_f1'] - tab.loc['P3', 'p_f1']
            speed = tab.loc['P3', 't_total'] / tab.loc[lv, 't_total']
            print(f'{lv:5s} {d_f1:+6.2f} diem p_f1  doi lay {speed:.2f}x  '
                  f'{"(vuot moc >=3x cua spec muc 7)" if speed >= 3 else ""}')